# Bearing fault diagnosis with clean and noisy STFT and DWT features

This notebook adapts the instructor's 1D-CNN to classify healthy, inner-race, ball and outer-race vibrations. It reads numeric STFT and DWT features for both clean and noise-added copies.

Install the project's dependencies from the repository root with `python -m pip install -r requirements.txt`. The first preparation run builds [Datasets/Features](../Datasets/Features) from the checked-in raw recordings. You can also extract the prepared dataset into `Datasets` before running the notebook.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "Datasets" / "RAW").is_dir():
    raise RuntimeError("Open Jupyter from the repository root or notebooks folder.")
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

from src.feature_dataset import (
    CLASS_FOLDERS, FEATURES_DIR, RAW_DIR, VARIANTS, export_dataset,
    load_partitions, read_array, read_index,
)
from src.stft_dwt import DWT_BANDS, FRAME_HOP, FRAME_SAMPLES, SAMPLE_RATE
from src.train_cnn import MODEL_PATH, evaluate_test, fit_model

## 1. Build and load the four-class dataset

Every folder has paired clean and noisy waveform, STFT and DWT arrays, with one shared index mapping the pairs to the original recordings. The default noise is additive white Gaussian noise at 10 dB per recording with seed 42. The 0 and 1 HP groups train the model, 2 HP groups validate it, and 3 HP groups test it.

In [ ]:
dataset_dir = FEATURES_DIR
if not (dataset_dir / "dataset.json").is_file():
    export_dataset(RAW_DIR, dataset_dir)
partitions = load_partitions(dataset_dir)

for split, variants in partitions.items():
    for variant, (features, labels, groups) in variants.items():
        print(split, variant, features.shape, np.unique(groups).size, "recordings")

Each 2,048-sample window produces 15 time frames. Each frame contributes 129 Hann STFT log-power bins and five db4 DWT log mean-square bands. The figures compare the two variants of the same validation window for each class. Each pair shares color scales for its STFT and DWT panels.

In [ ]:
def plot_feature_pair(folder):
    rows = read_index(folder)
    index = next(i for i, row in enumerate(rows) if row["partition"] == "validation")
    arrays = {
        name: [read_array(folder / variant, name)[index] for variant in VARIANTS]
        for name in ("signal", "stft", "dwt")
    }
    waves = arrays["signal"]
    samples_ms = np.arange(waves[0].size) * 1000 / SAMPLE_RATE
    frame_ms = (FRAME_SAMPLES / 2 + FRAME_HOP * np.arange(arrays["stft"][0].shape[0])) * 1000 / SAMPLE_RATE
    frequency_khz = np.fft.rfftfreq(FRAME_SAMPLES, d=1 / SAMPLE_RATE) / 1000
    bands = np.arange(len(DWT_BANDS))
    fig, axes = plt.subplots(3, 2, figsize=(10, 7), sharey="row", layout="constrained")
    fig.suptitle(f"{folder.name} · {rows[index]['record_id']} · paired validation window")

    for column, variant in enumerate(VARIANTS):
        wave_ax, stft_ax, dwt_ax = axes[:, column]
        wave_ax.plot(samples_ms, waves[column], lw=0.7)
        wave_ax.set_title(f"{variant} signal")
        stft = stft_ax.pcolormesh(
            frame_ms, frequency_khz, arrays["stft"][column].T, shading="auto",
            vmin=np.min(arrays["stft"]), vmax=np.max(arrays["stft"]),
        )
        stft_ax.set_title("STFT log power")
        stft_ax.set_ylim(0, SAMPLE_RATE / 2000)
        dwt = dwt_ax.pcolormesh(
            frame_ms, bands, arrays["dwt"][column].T, shading="auto",
            vmin=np.min(arrays["dwt"]), vmax=np.max(arrays["dwt"]),
        )
        dwt_ax.set_title("DWT log mean squared coefficients")
        dwt_ax.set_yticks(bands, DWT_BANDS)
        dwt_ax.set_xlabel("Time (ms)")

    for ax, label in zip(axes[:, 0], ("Amplitude", "Frequency (kHz)", "Wavelet band")):
        ax.set_ylabel(label)
    fig.colorbar(stft, ax=axes[1, :], label="dB")
    fig.colorbar(dwt, ax=axes[2, :], label="dB")
    return fig

for folder_name in CLASS_FOLDERS.values():
    plot_feature_pair(dataset_dir / folder_name)
    plt.show()

## 2. Train on both variants

The model keeps the instructor's two Conv1D and MaxPooling1D blocks. Feature normalization is fitted on the clean and noisy training windows. Weight selection uses the clean and noisy validation windows.

In [ ]:
model, history = fit_model(partitions)
model.summary()

## 3. Evaluate each condition on held-out recordings

The test uses ten recordings at 3 HP. Clean and noisy results are reported separately for the same ten recordings. For recording-level predictions, the model averages its window probabilities within each recording.

In [ ]:
for variant, partition in partitions["test"].items():
    metrics = evaluate_test(model, partition)
    print(variant)
    for unit in ("windows", "recordings"):
        result = metrics[unit]
        print(f"  {unit}: n={result['count']}, accuracy={result['accuracy']:.3f}, macro-F1={result['macro_f1']:.3f}")

    display = ConfusionMatrixDisplay(
        confusion_matrix=np.asarray(metrics["recordings"]["confusion_matrix"]),
        display_labels=metrics["class_order"],
    ).plot(xticks_rotation=30, cmap="Blues")
    display.ax_.set_title(f"{variant} held-out recordings")
    display.figure_.tight_layout()
    plt.show()

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)
print(f"Saved {MODEL_PATH}")